In [17]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time
import threading
import csv
import os
from datetime import datetime
from twilio.rest import Client
import pygame
import geocoder

# Initialize Face Mesh and drawing utils
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)
mp_drawing = mp.solutions.drawing_utils

# EAR Thresholds
EAR_THRESHOLD = 0.25
CONSEC_FRAMES = 20
frame_counter = 0
alarm_on = False
call_made = False
drowsiness_start_time = None  # Track the time when drowsiness is first detected

# Initialize pygame mixer for alarm
pygame.mixer.init()
alarm_sound = "alarm.wav"

# EAR calculation from eye landmarks
def calculate_EAR(landmarks, eye_indices, image_width, image_height):
    coords = [(int(landmarks[i].x * image_width), int(landmarks[i].y * image_height)) for i in eye_indices]
    A = math.dist(coords[1], coords[5])
    B = math.dist(coords[2], coords[4])
    C = math.dist(coords[0], coords[3])
    ear = (A + B) / (2.0 * C)
    return ear

# Alarm thread function
def play_alarm():
    pygame.mixer.music.load(alarm_sound)
    pygame.mixer.music.play(-1)

# Stop alarm
def stop_alarm():
    pygame.mixer.music.stop()

# CSV file setup
csv_file = "drowsiness_log.csv"
if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Date", "Time", "EAR", "Status", "Location", "Coordinates"])

# Call using Twilio and log to CSV
def send_alert_sms_and_call_with_location():
    account_sid = 'AC9098514e02fd29ba1abcf635eab3de29'
    auth_token = '6038f063e21abcd467a0e9d2563359b1'
    client = Client(account_sid, auth_token)

    # Step 1: Get Location
    g = geocoder.ip('me')  # Uses your public IP
    location = g.city + ", " + g.country if g.ok else "Unknown Location"
    latlng = g.latlng if g.ok else ["N/A", "N/A"]

    # Step 2: Format Message
    now = datetime.now()
    location_link = f"https://www.google.com/maps?q={latlng[0]},{latlng[1]}"
    body = f"⚠️ Drowsiness detected at {now.strftime('%H:%M:%S on %d-%m-%Y')}.\nLocation: {location}\nMap: {location_link}"

    try:
        # Step 3: Send SMS
        message = client.messages.create(
            body=body,
            from_='+1 947 219 0384',
            to='+917368052171'
        )
        print("SMS with location sent:", message.sid)

        # Step 4: Make Call
        call = client.calls.create(
            url='https://handler.twilio.com/twiml/EHac3b3355397649bc7575bbdb3b7afca4',
            to='+917368052171',
            from_='+1 947 219 0384'
        )
        print("Call initiated:", call.sid)

        # Step 5: Log to CSV
        with open(csv_file, mode="a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow([
                now.date(),
                now.strftime("%H:%M:%S"),
                "Below Threshold",
                "Drowsiness Detected",
                location,
                f"{latlng[0]},{latlng[1]}"
            ])
            print(f"[LOGGED] Drowsiness at {now.strftime('%H:%M:%S')} on {now.date()}")
            print(f"[LOCATION] {location} ({latlng[0]}, {latlng[1]})")

    except Exception as e:
        print("Alert failed:", e)

# Start webcam
cap = cv2.VideoCapture(0)

# Eye landmark indices for EAR calculation
LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]

while True:
    success, frame = cap.read()
    if not success:
        break

    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(rgb)

    if result.multi_face_landmarks:
        for face_landmarks in result.multi_face_landmarks:
            left_ear = calculate_EAR(face_landmarks.landmark, LEFT_EYE, w, h)
            right_ear = calculate_EAR(face_landmarks.landmark, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            cv2.putText(frame, f"EAR: {avg_ear:.2f}", (30, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

            if avg_ear < EAR_THRESHOLD:
                if drowsiness_start_time is None:
                    drowsiness_start_time = time.time()

                elapsed_time = time.time() - drowsiness_start_time

                if elapsed_time >= 3 and not alarm_on:
                    alarm_on = True
                    threading.Thread(target=play_alarm).start()

                    if not call_made:
                        call_made = True
                        threading.Thread(target=send_alert_sms_and_call_with_location).start()

                if elapsed_time >= 3:
                    cv2.putText(frame, "DROWSINESS DETECTED!!", (30, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            else:
                drowsiness_start_time = None
                alarm_on = False
                call_made = False
                stop_alarm()

    cv2.imshow("Drowsiness Detection", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
        break

cap.release()
cv2.destroyAllWindows()


SMS with location sent: SMb81f192e4bc45b2b8b535c95d95ad00f
Call initiated: CA5104b14aad992537b876f2e3f38e5abf
[LOGGED] Drowsiness at 17:23:50 on 2025-04-23
[LOCATION] Chennai, IN (13.0878, 80.2785)
